# Pick `min_corr` / `min_pnr` thresholds

Run the cells below **after motion correction has finished** (you need the motion-corrected `.mmap` file).
The interactive sliders let you visually identify good `min_corr` and `min_pnr` values to pass to `preproc_caiman_final.py`.

In [ ]:
%matplotlib widget

import numpy as np
import os

try:
    import cv2
    cv2.setNumThreads(0)
except Exception:
    pass

os.environ['CAIMAN_DATA'] = "/media/toor/Seagate3/caiman_data"

import caiman as cm
from caiman.utils.visualization import inspect_correlation_pnr

In [ ]:
# Already-cropped mmap written by preproc_caiman_final.py — no need to crop again
MMAP_PATH = "/media/toor/Seagate3/caiman_data/temp/memmap_cropped_d1_210_d2_468_d3_1_order_C_frames_2250.mmap"

GSIG = 5   # must match gSig used in preproc_caiman_final.py

Yr, dims, num_frames = cm.load_memmap(MMAP_PATH)
images = np.reshape(Yr.T, [num_frames] + list(dims), order='F')

img_min = images.min()
if img_min < 0:
    images -= img_min

print(f"Loaded {num_frames} frames, shape: {images.shape}")

## Compute correlation and PNR images

In [ ]:
cn, pnr = cm.summary_images.correlation_pnr(
    images[::max(num_frames // 1000, 1)],
    gSig=GSIG,
    swap_dim=False,
)
print(f"Correlation image  —  min: {cn.min():.3f}  max: {cn.max():.3f}")
print(f"PNR image          —  min: {pnr.min():.1f}  max: {pnr.max():.1f}")

## Interactive threshold picker

Drag the sliders until the seeds (overlaid dots) match the cell bodies visible in the correlation image.
Note down the `min_corr` and `min_pnr` values and pass them to `preproc_caiman_final.py`.

In [ ]:
inspect_correlation_pnr(cn, pnr)